# 02 - Reversible training at the SAME batch size: which variant works best?

**Goal (assignment part 2).** Re-train with a *reversible* transformer at the batch size fixed in
notebook 01, and compare the reversible update rules:

| variant | update rule | note |
|---|---|---|
| `midpoint` | `x[l+1] = x[l-1] + 2h f(x[l])` | 2-step rule from the paper |
| `leapfrog` | `x[l+1] = 2x[l] - x[l-1] + h^2 f(x[l])` | 2nd-order, wave-equation style |
| `hamiltonian` | `q' = q + Attn(p);  p' = p + MLP(q')` | symplectic-Euler ("Euler-style") |

All variants have exactly the same parameters as the baseline. Report which one wins on loss.

In [ ]:
SMOKE = False   # True = tiny CPU version of this whole notebook (used to test the code; not for results)
REPO_URL = "https://github.com/YOUR-USERNAME/reversible-llm-20m"   # <- edit: only needed on Colab
USE_DRIVE = False   # True = keep data + results on Google Drive so they survive a Colab disconnect

import os, sys, copy, json, subprocess
if os.path.exists("../revlm"):
    os.chdir("..")                                   # opened from the notebooks/ folder
elif not os.path.exists("revlm"):
    assert "YOUR-USERNAME" not in REPO_URL, "edit REPO_URL (top of this cell) to point at your GitHub repo"
    subprocess.run(["git", "clone", REPO_URL, "repo"], check=True)   # fresh Colab VM
    os.chdir("repo")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "tokenizers", "datasets"], check=False)

import torch
from IPython.display import Image, display
from revlm import selftest, experiment as ex
from revlm.bench import bench_steps, scaling_table
from revlm.data import prepare_data, Data
from revlm.report import write_summary, plot_scaling

S = ex.settings(SMOKE)
DEV = ex.device()
RESULTS = "results_smoke" if SMOKE else "results"
DATA_DIR = "data_smoke" if SMOKE else "data"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS = "/content/drive/MyDrive/reversible-llm-20m/" + RESULTS
    DATA_DIR = "/content/drive/MyDrive/reversible-llm-20m/" + DATA_DIR
print("device:", DEV, torch.cuda.get_device_name() if DEV == "cuda" else "(no GPU!)" if not SMOKE else "(smoke test)")
if DEV != "cuda" and not SMOKE:
    print("WARNING: no GPU. In Colab use Runtime > Change runtime type > GPU.")

## 0. Sanity check

In [ ]:
# 30-second correctness check on THIS machine/GPU before spending an hour of compute.
assert selftest.main(DEV), "self-test failed - do not trust any numbers below"

## 1. Data

In [ ]:
# Downloads TinyStories, trains an 8k BPE tokenizer, writes exactly 50M training tokens (~3-6 min, cached).
prepare_data(DATA_DIR, vocab_size=8192, **S["data"])
data = Data(DATA_DIR, S["model"].ctx, seed=1337)
print(f"train tokens available: {len(data.train):,}   validation tokens: {len(data.val):,}   sequences: {data.n_seq:,}")

In [ ]:
def mk(mode, **over):
    """20M-parameter model config in the given mode (same weights/shape for every mode)."""
    mc = copy.deepcopy(S["model"]); mc.mode = mode
    for k, v in over.items():
        setattr(mc, k, v)
    return mc

from revlm.model import GPT
print(f"parameters: {GPT(mk('baseline')).num_params()/1e6:.2f}M   (layers={S['model'].n_layer}, d_model={S['model'].d_model}, vocab={S['model'].vocab_size}, ctx={S['model'].ctx})")

In [ ]:
saved = ex.load(RESULTS, "fixed_batch")
B_FIXED = saved["B_fixed"] if saved else None      # <- if you skipped notebook 01, type the batch size here
assert B_FIXED, "run notebook 01 first (or set B_FIXED by hand)"
print("using batch size", B_FIXED)

## 2. Quick speed / memory check at the fixed batch (5 real training steps each)

In [ ]:
bench = {}
for mode in ["baseline", "midpoint", "leapfrog", "hamiltonian"]:
    r = bench_steps(mk(mode), B_FIXED, DEV, "auto", n_steps=6)
    bench[mode] = r
    print(f"{mode:12s} ok={r['ok']}  peak={r['peak_mib']}  tok/s={r['tokens_per_sec']}")
ex.save(RESULTS, "bench_fixed_batch", bench)

## 3. Train all three reversible variants for 50M tokens

In [ ]:
variants = {}
for mode in ["midpoint", "leapfrog", "hamiltonian"]:
    variants[mode] = ex.run_or_load(mk(mode), ex.make_train_cfg(S, f"02_{mode}", B_FIXED), data, RESULTS)

## 4. Optional: does the step size `h` matter? (short pilot, off by default)
`h` is the step size of midpoint / leapfrog. The default `h = 0.5` makes midpoint's step `2h = 1`,
the same size as the baseline's residual step. Turn this on to test 0.25 / 0.5 / 1.0 on 5M tokens.

In [ ]:
RUN_H_SWEEP = False
if RUN_H_SWEEP:
    S_pilot = dict(S, total_tokens=5_000_000 if not SMOKE else 200_000, eval_every_tokens=2_500_000 if not SMOKE else 100_000)
    for mode in ["midpoint", "leapfrog"]:
        for h in [0.25, 0.5, 1.0]:
            tc = ex.make_train_cfg(S_pilot, f"hsweep_{mode}_h{h}", B_FIXED)
            r = ex.run_or_load(mk(mode, h=h), tc, data, RESULTS + "/hsweep")
            print(mode, h, "val loss", round(r["final_val_loss"], 4), "diverged" if r["diverged"] else "")

## 5. Which variant wins?

In [ ]:
base = ex.load(RESULTS, "01_baseline")
ok = {m: r for m, r in variants.items() if not r["diverged"] and r["final_val_loss"] == r["final_val_loss"]}
winner = min(ok, key=lambda m: ok[m]["final_val_loss"])
ex.save(RESULTS, "winner", {"winner": winner, "h": ok[winner]["model"]["h"]})
print(f"{'run':14s}{'val loss':>10s}{'tok/s':>10s}{'peak MiB':>10s}")
for name, r in [("baseline", base)] + list(variants.items()):
    if r: print(f"{name:14s}{r['final_val_loss']:10.4f}{r['tokens_per_sec_median']:10,.0f}{(r['peak_mem_allocated_mib'] or 0):10,.0f}")
print("winner (lowest validation loss):", winner)

## 6. Does memory really stop growing with depth? (baseline vs the winner)
Fixed batch, more and more layers, then longer sequences. Reversible memory should stay almost flat
in depth; the baseline's should grow linearly.

In [ ]:
variants_cfg = [("baseline", {"mode": "baseline"}), (winner, {"mode": winner})]
print("depth scaling"); rows_d = scaling_table(S["model"], variants_cfg, DEV, "auto", S["bench_batch"], "n_layer", S["depths"])
print("sequence-length scaling"); rows_c = scaling_table(S["model"], variants_cfg, DEV, "auto", max(1, S["bench_batch"] // 2), "ctx", S["ctxs"])
ex.save(RESULTS, "depth_scaling", {"rows": rows_d}); ex.save(RESULTS, "ctx_scaling", {"rows": rows_c})
if DEV == "cuda":
    plot_scaling(rows_d, "n_layer", f"{RESULTS}/depth_scaling.png", "number of transformer blocks")
    plot_scaling(rows_c, "ctx", f"{RESULTS}/ctx_scaling.png", "sequence length")
    display(Image(f"{RESULTS}/depth_scaling.png")); display(Image(f"{RESULTS}/ctx_scaling.png"))

## 7. Loss curves and summary table

In [ ]:
write_summary(RESULTS)
display(Image(f"{RESULTS}/loss_curves.png")); display(Image(f"{RESULTS}/speed_memory.png"))